# Jina Reranker

Jina Reranker 是一个用于对搜索结果进行重新排序的组件。

## 安装

```bash
pip install jina-reranker
```

## 用法

```python
from jina.executors.rankers import Ranker
from jina.proto import jina_pb2

class MyRanker(Ranker):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def _apply_all(self, query_chunk, match_chunks, *args, **kwargs):
        # 在这里实现你的重新排序逻辑
        # query_chunk 是一个 Document 对象，代表查询
        # match_chunks 是一个 Document 对象列表，代表搜索结果
        # 返回一个 Document 对象列表，代表重新排序后的搜索结果
        return match_chunks

from jina import Flow, Document

# 创建一个 Flow
f = Flow().add(uses=MyRanker)

# 运行 Flow
with f:
    f.post(Document(text="query"), on="/search")
```

## 自定义重新排序器

你可以通过继承 `Ranker` 类并实现 `_apply_all` 方法来创建自己的重新排序器。

`_apply_all` 方法接收以下参数：

- `query_chunk`: 一个 `Document` 对象，代表查询。
- `match_chunks`: 一个 `Document` 对象列表，代表搜索结果。

该方法应返回一个 `Document` 对象列表，代表重新排序后的搜索结果。

## 示例

以下是一个简单的示例，演示如何使用 Jina Reranker 对搜索结果按相关性得分进行重新排序：

```python
from jina.executors.rankers import Ranker
from jina.proto import jina_pb2

class RelevanceRanker(Ranker):
    def _apply_all(self, query_chunk, match_chunks, *args, **kwargs):
        # 按相关性得分降序排序
        match_chunks.sort(key=lambda x: x.score, reverse=True)
        return match_chunks

from jina import Flow, Document

# 创建一个 Flow
f = Flow().add(uses=RelevanceRanker)

# 运行 Flow
with f:
    f.post(Document(text="query"), on="/search")

本 Notebook 展示了如何使用 Jina Reranker 进行文档压缩和检索。

In [ ]:
%pip install -qU langchain langchain-openai langchain-community langchain-text-splitters langchainhub

%pip install --upgrade --quiet  faiss

# OR  (depending on Python version)

%pip install --upgrade --quiet  faiss_cpu

In [ ]:
# Helper function for printing docs


def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

## 设置基础向量存储检索器

让我们通过初始化一个简单的向量存储检索器，并存储 2023 年国情咨文演讲（分块）。我们可以将检索器设置为检索大量（20 个）文档。

##### 设置 Jina 和 OpenAI API 密钥

In [ ]:
import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass.getpass()
os.environ["JINA_API_KEY"] = getpass.getpass()

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = TextLoader(
    "../../how_to/state_of_the_union.txt",
).load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = text_splitter.split_documents(documents)

embedding = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")
retriever = FAISS.from_documents(texts, embedding).as_retriever(search_kwargs={"k": 20})

query = "What did the president say about Ketanji Brown Jackson"
docs = retriever.get_relevant_documents(query)
pretty_print_docs(docs)

## 使用 JinaRerank 进行重排

JinaRerank 是一个用于对搜索结果进行重排的工具。它能够根据用户查询和文档内容，对搜索结果的顺序进行优化，从而提供更相关的结果。

### 安装

您可以使用 pip 来安装 JinaRerank：

```bash
pip install jina-rerank
```

### 使用方法

以下是如何使用 JinaRerank 对搜索结果进行重排的示例：

假设您有一个查询和一个文档列表：

```python
from jina.parsers.rerank import set_rerank_parser
from jina.types.document import Document
from jina.types.request import Request

# 创建一个查询
query = "What is the capital of China?"

# 创建一个文档列表
docs = [
    Document(content="The capital of China is Beijing."),
    Document(content="Shanghai is a major city in China."),
    Document(content="The Great Wall of China is a famous landmark."),
]

# 创建一个请求对象
req = Request()
req.search.query = query
for doc in docs:
    req.search.docs.append(doc)

# 设置重排解析器
args = set_rerank_parser().parse_args([])

# 使用 JinaRerank 进行重排
from jina.executors.rankers import Ranker
reranker = Ranker.load_executor(name="JinaRerank", runtime_args=args)
reranker.rank(req)

# 打印重排后的文档
for doc in req.search.docs:
    print(f"Score: {doc.scores['rerank'].value}, Content: {doc.content}")
```

### 工作原理

JinaRerank 使用一个预训练的语言模型来理解查询和文档之间的语义关系。它会为每个文档生成一个分数，分数越高表示文档与查询越相关。然后，它会根据这些分数对文档进行排序。

### 配置

您可以通过修改 `set_rerank_parser()` 返回的参数来配置 JinaRerank 的行为。例如，您可以指定要使用的模型、调整模型的参数等。

### 更多信息

有关 JinaRerank 的更多信息，请参阅官方文档：[https://docs.jina.ai/concepts/reranking/](https://docs.jina.ai/concepts/reranking/)

现在，让我们使用 Jina Reranker 作为压缩器，将我们的基础检索器包装在 ContextualCompressionRetriever 中。

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors import JinaRerank

compressor = JinaRerank()
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

compressed_docs = compression_retriever.get_relevant_documents(
    "What did the president say about Ketanji Jackson Brown"
)

In [ ]:
pretty_print_docs(compressed_docs)

## 使用 Jina Reranker 进行 QA 重排

Jina Reranker 是一个用于对搜索结果进行重排的工具。它通过一个预训练的 Transformer 模型来衡量查询和文档之间的相关性，并根据相关性得分对文档进行排序。

### 安装

您可以使用 pip 安装 Jina Reranker：

```bash
pip install jina-reranker
```

### 使用方法

以下是如何使用 Jina Reranker 对搜索结果进行重排的示例：

```python
from jina.proto import jina_pb2
from jina.parsers.base import set_parsers
from jina.flow import Flow

# 假设您有一个查询和一个文档列表
query = "What is the capital of France?"
documents = [
    "The capital of France is Paris.",
    "France is a country in Europe.",
    "Paris is known for the Eiffel Tower.",
]

# 创建一个 Flow
with Flow() as flow:
    # 添加一个 Reranker 节点
    flow.add(
        uses="jina-reranker",
        name="reranker",
        port_expose=12345,
        replicas=1,
    )

    # 运行 Flow
    with flow.build():
        # 发送查询和文档到 Reranker 节点
        response = flow.post(
            on="/search",
            inputs=[
                jina_pb2.Document(
                    id="query",
                    content=query,
                    tags={"type": "query"},
                ),
                *[
                    jina_pb2.Document(id=f"doc_{i}", content=doc)
                    for i, doc in enumerate(documents)
                ],
            ],
        )

        # 打印重排后的文档
        for doc in response.docs[0].matches:
            print(f"Score: {doc.score.value}, Content: {doc.content}")
```

在此示例中，我们首先定义了一个查询和一组文档。然后，我们创建一个 Jina Flow 并添加一个 Reranker 节点。最后，我们将查询和文档发送到 Reranker 节点，并打印重排后的文档。

Jina Reranker 可以用于各种场景，例如：

- **信息检索：** 对搜索结果进行重排，以提高搜索准确性。
- **问答系统：** 对候选答案进行重排，以找到最相关的答案。
- **推荐系统：** 对推荐项目进行重排，以提高用户满意度。

Jina Reranker 是一个强大而灵活的工具，可以帮助您提高搜索结果的质量。

In [1]:
from langchain import hub
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

retrieval_qa_chat_prompt = hub.pull("langchain-ai/retrieval-qa-chat")
retrieval_qa_chat_prompt.pretty_print()

================================ System Message ================================

Answer any use questions based solely on the context below:

<context>
{context}
</context>

============================= Messages Placeholder =============================

{chat_history}

================================ Human Message =================================

{input}


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
combine_docs_chain = create_stuff_documents_chain(llm, retrieval_qa_chat_prompt)
chain = create_retrieval_chain(compression_retriever, combine_docs_chain)

In [ ]:
chain.invoke({"input": query})